
# K-Means: Kaggle Customer Segmentation Dataset

Dataset:
kaggle kernels pull karnikakapoor/customer-segmentation-clustering

İçerik:
- Veri yükleme
- Ön işleme
- Elbow Method
- Silhouette analiz
- Çoklu k animasyon (GIF)
- WCSS + Silhouette overlay


In [ ]:

# Gerekli kütüphaneler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from PIL import Image
import os


In [ ]:

# Veri setini yükle
# Kaggle'dan indirildikten sonra CSV yolu güncellenmeli

df = pd.read_csv("Mall_Customers.csv")

df.head()


In [ ]:

# Feature seçimi
X = df[["Annual Income (k$)", "Spending Score (1-100)"]]

# Ölçekleme
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:

# Elbow Method
wcss = []
k_range = range(2, 10)

for k in k_range:
    model = KMeans(n_clusters=k, n_init=10)
    model.fit(X_scaled)
    wcss.append(model.inertia_)

plt.figure()
plt.plot(list(k_range), wcss, marker='o')
plt.title("Elbow Method")
plt.xlabel("k")
plt.ylabel("WCSS")
plt.savefig("elbow.png")
plt.close()


In [ ]:

# Animasyon klasörü
os.makedirs("frames", exist_ok=True)

k_values = [2,3,4,5]
iterations = 8
frames = []

for i in range(iterations):
    fig, axes = plt.subplots(1, 2, figsize=(10,4))

    # clustering plot
    for k in k_values:
        model = KMeans(n_clusters=k, n_init=1, max_iter=i+1)
        labels = model.fit_predict(X_scaled)

        axes[0].scatter(X_scaled[:,0], X_scaled[:,1], c=labels)

        sil = silhouette_score(X_scaled, labels)
        axes[1].bar(k, sil)

    axes[0].set_title(f"Clustering iter={i}")
    axes[1].set_title("Silhouette Scores")
    axes[1].set_ylim(0,1)

    path = f"frames/frame_{i}.png"
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

    frames.append(Image.open(path))

frames[0].save("kmeans_kaggle.gif", save_all=True,
               append_images=frames[1:], duration=700, loop=0)
